# MoleOOD Shared Prompt A
启用 GPU 后依次运行。源域准备后比较原模型、冻结提示、episodic 与 continual。数据不在 GitHub，需要放入自己的 Drive。目标测试标签不用于更新或调参。

In [ ]:
!git clone --branch prompt-a --single-branch https://github.com/qwqwqwqwqwqwqwqwqe/MoleOOD.git /content/MoleOOD
%cd /content/MoleOOD
!git log -1 --oneline

In [ ]:
import sys, subprocess, torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "请在 Colab 设置中选择 GPU"
subprocess.check_call([sys.executable, "-m", "pip", "install", "torch-geometric", "rdkit", "scikit-learn", "pytest"])
base = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
cuda = "cu" + torch.version.cuda.replace(".", "")
wheel_url = f"https://data.pyg.org/whl/torch-{base}+{cuda}.html"
print("PyG wheel index:", wheel_url)
# 不从源码编译扩展。如果当前 Colab 的 PyTorch/CUDA 没有匹配 wheel，此处会报错。
subprocess.check_call([sys.executable, "-m", "pip", "install", "torch-scatter", "--only-binary=:all:", "-f", wheel_url])
import torch_geometric, torch_scatter, rdkit
print("Graph dependencies ready")

In [ ]:
!python -m pytest tests/test_prompt_a.py -q
!python prompt_a.py --help

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DATA = Path("/content/drive/MyDrive/MoleOOD/lbap_core_ec50_scaffold_brics.json") # 修改为你的数据路径
OUTPUT = Path("/content/drive/MyDrive/MoleOOD/prompt_a/seed2022")
MODEL = Path("/content/MoleOOD/log/lbap_core_ec50_scaffold_brics/PyG_GIN/best_model.pth")
assert DATA.is_file(), f"找不到数据: {DATA}"
assert MODEL.is_file(), f"找不到权重: {MODEL}"

In [ ]:
subprocess.check_call([sys.executable, "prompt_a.py", "--stage", "all",
    "--model-path", str(MODEL), "--train-path", str(DATA), "--test-path", str(DATA),
    "--device", "cuda:0", "--seed", "2022", "--output", str(OUTPUT)])

In [ ]:
import json, pandas as pd
result = json.loads((OUTPUT / "results.json").read_text())
display(pd.DataFrame(result["results"]).T)
print("结果已保存到", OUTPUT)

## 结果解释
先看 source_frozen 对 moleood 的差异，再看适应组对 source_frozen 的差异。建议固定配置再运行种子 2023、2024，修改对应输出目录。不要根据测试 AUC 搜索超参数。

本 notebook 未在 Colab 实机验证；若 torch-scatter 无匹配 wheel，停止并核对上方 PyTorch/CUDA 版本，不要盲目源码编译。完整说明见 PROMPT_A.md。